# Bato: flood damage and cut-off villages from space
Run every cell from top to bottom (Runtime > Run all). No install on your laptop needed.

*Satellites measure, code counts, AI only words it.*

In [ ]:
# 1. Get the code (change the URL to your GitHub repo)
REPO = 'https://github.com/irfanix/bato-flood'
import os, glob
%cd /content
!rm -rf bato-flood
!git clone -q $REPO bato-flood
root = os.path.dirname(sorted(glob.glob('/content/bato-flood/**/requirements.txt', recursive=True), key=len)[0])
%cd $root
!grep -c "overpass" bato/osm.py

In [ ]:
# 2. Install (about 1 minute)
!pip install -q -r requirements.txt

In [ ]:
# 3. Run the tests (no internet data needed)
!python -m pytest -q tests -W ignore

In [ ]:
# 4. CASE STUDY: Trishuli flood, 26 Aug 2026, from raw Sentinel-1/2 + Copernicus DEM + pre-event OSM
#    Also traces the flood path from just below the Rasuwagadhi border (bonus)
!python -m bato run --preset trishuli --out case_study_trishuli --flow 85.375 28.27

In [ ]:
# 5. Look at the map
from IPython.display import HTML, display
display(HTML(open('case_study_trishuli/map.html', encoding='utf-8').read()))

In [ ]:
# 6. One-page situation report (English + Nepali)
display(HTML(open('case_study_trishuli/situation_report.html', encoding='utf-8').read()))

## 7. Check against Copernicus EMS EMSR927 (validation only, never an input)
Download the EMSR927 vector package (.zip) from mapping.emergency.copernicus.eu, then run the next cell and upload it.

In [ ]:
from google.colab import files
up = files.upload()
zips = ' '.join(up.keys())
!python -c "import json,sys; from bato.pipeline import load_dir; from bato.compare import compare; R=load_dir('case_study_trishuli'); r=compare(R.cls,R.grid,sys.argv[1:],None,'case_study_trishuli/ems_comparison.png'); json.dump(r,open('case_study_trishuli/ems_comparison.json','w'),indent=1); print(json.dumps({k:v for k,v in r.items() if k!='ems_layers'},indent=1))" $zips
from IPython.display import Image
Image('case_study_trishuli/ems_comparison.png')

In [ ]:
# 8. Live test on ANY area and date (what the judges will do). Example: change bbox/date
!python -m bato run --bbox 85.30 28.10 85.40 28.20 --date 2026-08-26 --res 30 --out runs/test_area

In [ ]:
# 9. Download the case-study folder, unzip it into your repo, and commit it with GitHub Desktop
!rm -f case_study_trishuli/ems_upload.zip
!zip -qr case_study_trishuli.zip case_study_trishuli
files.download('case_study_trishuli.zip')